# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and contains clinicopathological variables for cancer survivors with second primary colorectal cancer.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their corresponding `@id`s.

We'll inspect the `recordSet` attribute to enumerate the available record sets.

Fields and columns are referenced by their `@id` for clear, consistent access.

In [ ]:
# List record set IDs
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        if hasattr(rs, '@id'):
            print(f"RecordSet @id: {rs['@id']}")
            record_sets.append(rs['@id'])

# Print out record sets and their fields
for rsid in record_sets:
    print(f"\nRecordSet: {rsid}")
    rs_obj = dataset.metadata.get_by_id(rsid)
    if hasattr(rs_obj, 'field') and rs_obj.field:
        for f in rs_obj.field:
            fid = f['@id'] if isinstance(f, dict) and '@id' in f else f
            print(f"  Field @id: {fid}")
            # Try to get column(s) info from the field
            field_obj = dataset.metadata.get_by_id(fid)
            if hasattr(field_obj, 'column') and field_obj.column:
                for c in field_obj.column:
                    cid = c['@id'] if isinstance(c, dict) and '@id' in c else c
                    print(f"    Column @id: {cid}")

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use record set and field/column `@id`s from the overview.

We'll extract all records from each record set into a pandas DataFrame.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nDataFrame for RecordSet {record_set_id}:")
    print(df.columns.tolist())
    print(df.head(3))

# For demonstration, select the primary RecordSet if available
primary_record_set_id = record_sets[0] if record_sets else None
if primary_record_set_id:
    print(f"\nPreview of records in {primary_record_set_id}:")
    print(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll demonstrate how to filter for patients above a certain age threshold, normalize the age, and group by anatomical location.

**Note:** All field and column references are via their `@id`.

In [ ]:
# Identify numeric and group fields by @id (edit these per actual schema IDs)
# Example: Assume field for age has @id 'http://mlcommons.org/croissant/age',
# and anatomical location column @id 'http://mlcommons.org/croissant/anatomical_location'

# Find available columns in primary record set
df = dataframes.get(primary_record_set_id, pd.DataFrame())
print("Available columns:", df.columns.tolist())

# Suppose age field has @id 'http://mlcommons.org/croissant/age' and is present
numeric_field_id = None
possible_age_ids = ['age', 'http://mlcommons.org/croissant/age']
for col in df.columns:
    if col.lower() in possible_age_ids or col.endswith('age'):
        numeric_field_id = col
        break

# Suppose group field anatomical location @id is 'http://mlcommons.org/croissant/anatomical_location'
group_field_id = None
possible_loc_ids = ['anatomical_location', 'http://mlcommons.org/croissant/anatomical_location']
for col in df.columns:
    if col.lower() in possible_loc_ids or col.endswith('anatomical_location'):
        group_field_id = col
        break

# EDA: filter patients older than 60
if numeric_field_id and numeric_field_id in df.columns:
    threshold = 60
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by anatomical location
    if group_field_id and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped.head())
else:
    print("No numeric field matching age found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we plot age distribution and mean age by anatomical location.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Age distribution
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel("Age")
    plt.ylabel("Count")
    plt.show()

# Mean age by anatomical location (if available)
if numeric_field_id and group_field_id and numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(10,4))
    group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
    group_means.plot.bar()
    plt.title("Mean Age by Anatomical Location")
    plt.ylabel("Mean Age")
    plt.xlabel("Anatomical Location")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR^2 dataset using `mlcroissant`. We demonstrated how to access and reference entities via their `@id`, extract records into pandas DataFrames, filter and normalize numeric variables, and visualize key clinical characteristics.

This workflow can be repeated for any Croissant-based dataset by consulting its schema, referencing entity `@id`s, and leveraging the `mlcroissant` library for reproducible FAIR data exploration.